# Disaster-Geomapper: Pipeline Walkthrough

This notebook runs each stage of `geolocate.py`'s pipeline one at a time and visualizes the intermediate output, so you can see *how* a prediction was arrived at rather than just the final answer.

It imports the real functions from `geolocate.py` / `ocr_match.py` / `exif_gps.py` / `mapvis.py` / `populated_places.py` — nothing here is reimplemented, this is purely a visualization harness over the actual pipeline code.

Stages covered:
1. Input image
2. EXIF GPS check
3. Raw GeoCLIP candidates (wide pool, before clustering)
4. Coarse-to-fine clustering + uncertainty
5. OCR scene-text detection
6. OCR × Overpass pairwise proximity refinement
7. Expanding-radius brand search
8. Populated-places snap (in progress — shows what's wired up so far)
9. Final ranked result

**Try a different image:** change `IMAGE_PATH` below. The bundled sample images (`images.jpg`, `bankier_032071323.webp`) have little to no legible signage, so stages 5-7 won't find much — swap in a street-level photo with visible storefront/brand text to see those stages actually fire.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from geoclip import GeoCLIP
from geolocate import (
    geolocate_image, cluster_candidates, _cluster_spread_km,
    refine_with_scene_text, analyze_image,
)
from exif_gps import get_exif_gps
from ocr_match import extract_text_candidates, _get_reader, find_brand_location_candidates
import mapvis
import populated_places

IMAGE_PATH = "images.jpg"

model = GeoCLIP()  # load once, reuse in every cell below

## 1. Input image

In [ ]:
img = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(IMAGE_PATH)
plt.show()

## 2. EXIF GPS check

If the photo still has GPS metadata, the real pipeline trusts it completely and skips everything below. Most web-sourced disaster photos won't have this (stripped on upload), so `None` here is the expected/common case.

In [ ]:
exif_coords = get_exif_gps(IMAGE_PATH)
print("EXIF GPS:", exif_coords)

## 3. Raw GeoCLIP candidates

Pulling a wide pool (`coarse_k=25`) rather than just the top-5, so the clustering step in the next section has something real to work with. Marker size/opacity reflects probability.

In [ ]:
import folium

coords, probs = geolocate_image(IMAGE_PATH, top_k=25, model=model)
coords = [(float(lat), float(lon)) for lat, lon in coords]
probs = [float(p) for p in probs]

for i, ((lat, lon), p) in enumerate(sorted(zip(coords, probs), key=lambda x: -x[1])[:5], start=1):
    print(f"{i}. {lat:.4f}, {lon:.4f}  (prob={p:.4f})")

avg_lat = sum(lat for lat, _ in coords) / len(coords)
avg_lon = sum(lon for _, lon in coords) / len(coords)
raw_map = folium.Map(location=[avg_lat, avg_lon], zoom_start=4, tiles="OpenStreetMap")
max_p = max(probs)
for (lat, lon), p in zip(coords, probs):
    folium.CircleMarker(
        location=[lat, lon], radius=4 + 10 * (p / max_p),
        color="crimson", fill=True, fill_opacity=0.6,
        tooltip=f"{lat:.4f}, {lon:.4f} (prob={p:.4f})",
    ).add_to(raw_map)
raw_map

## 4. Coarse-to-fine clustering

Groups the 25 raw candidates into spatial clusters. The dominant cluster (most probability mass) becomes GeoCLIP's real "coarse" guess. If there's more than one cluster, or the dominant one has low mass, that's GeoCLIP itself signaling uncertainty — which is exactly what caused the Champaign/Columbus misfire this pipeline was built to catch.

In [ ]:
clusters = cluster_candidates(coords, probs, radius_km=50.0)
dominant = clusters[0]
spread_km = _cluster_spread_km(dominant)

print(f"{len(clusters)} cluster(s) found")
for i, c in enumerate(clusters, start=1):
    marker = " <- dominant" if c is dominant else ""
    print(f"  cluster {i}: {len(c['members'])} point(s), mass={c['mass']:.1%}, centroid={c['centroid']}{marker}")
print(f"\nDominant cluster spread: ~{spread_km:.1f}km")

cluster_colors = ["red", "blue", "green", "purple", "orange", "darkred", "cadetblue"]
cluster_map = folium.Map(location=list(dominant["centroid"]), zoom_start=6, tiles="OpenStreetMap")
for i, c in enumerate(clusters):
    color = cluster_colors[i % len(cluster_colors)]
    for (lat, lon), p in c["members"]:
        folium.CircleMarker(
            location=[lat, lon], radius=5, color=color, fill=True, fill_opacity=0.7,
            tooltip=f"cluster {i+1}, prob={p:.4f}",
        ).add_to(cluster_map)
folium.Circle(
    location=list(dominant["centroid"]), radius=spread_km * 1000,
    color="black", fill=False, dash_array="5,5", tooltip="dominant cluster spread",
).add_to(cluster_map)
cluster_map

## 5. OCR scene-text detection

Raw EasyOCR output (every detected text region + confidence) drawn directly on the image, versus what actually survives `extract_text_candidates()`'s proper-noun/confidence filter and gets used downstream.

In [ ]:
reader = _get_reader()
ocr_results = reader.readtext(IMAGE_PATH)

annotated = img.copy()
draw = ImageDraw.Draw(annotated)
for bbox, text, conf in ocr_results:
    pts = [tuple(p) for p in bbox]
    draw.polygon(pts, outline="red", width=3)
    draw.text(pts[0], f"{text} ({conf:.2f})", fill="red")

plt.figure(figsize=(7, 7))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"Raw OCR: {len(ocr_results)} region(s) detected")
plt.show()

texts = extract_text_candidates(IMAGE_PATH)
print("Filtered candidates used by the pipeline:", texts)

## 6. OCR × Overpass pairwise proximity refinement

Needs >=2 detected names to do anything (it looks for pairs of named places that sit near each other in real life). Compares each fine candidate's raw GeoCLIP score to its score after this refinement pass.

In [ ]:
fine = sorted(dominant["members"], key=lambda m: m[1], reverse=True)[:5]
fine_coords = [pt for pt, _ in fine]
fine_probs = [p for _, p in fine]

refined = refine_with_scene_text(IMAGE_PATH, fine_coords, fine_probs, texts=texts, verbose=True)

labels = [f"#{i+1}" for i in range(len(fine_coords))]
before = fine_probs
after = [dict(refined)[c] for c in fine_coords]

x = range(len(labels))
plt.figure(figsize=(6, 4))
plt.bar([i - 0.2 for i in x], before, width=0.4, label="raw GeoCLIP")
plt.bar([i + 0.2 for i in x], after, width=0.4, label="after OCR refinement")
plt.xticks(list(x), labels)
plt.ylabel("score")
plt.legend()
plt.title("Score before/after pairwise OCR refinement")
plt.show()

## 7. Expanding-radius brand search

Works off a *single* distinctive name (no pair needed), searching outward from the dominant cluster's centroid until Overpass finds a fuzzy name/brand match. This is the stage that can override GeoCLIP entirely when it's confidently wrong about the whole region — shown as a line from GeoCLIP's guess to wherever the brand was actually found.

In [ ]:
brand_hits = find_brand_location_candidates(texts, anchor=dominant["centroid"]) if texts else []
print(f"{len(brand_hits)} brand match(es) found")

if brand_hits:
    brand_map = folium.Map(location=list(dominant["centroid"]), zoom_start=5)
    folium.Marker(list(dominant["centroid"]), tooltip="GeoCLIP's guess", icon=folium.Icon(color="red")).add_to(brand_map)
    for hit in brand_hits:
        folium.Marker(
            [hit["lat"], hit["lon"]], tooltip=f"{hit['name']} (matched '{hit['matched_text']}')",
            icon=folium.Icon(color="purple"),
        ).add_to(brand_map)
        folium.PolyLine([list(dominant["centroid"]), [hit["lat"], hit["lon"]]], color="gray", dash_array="5,5").add_to(brand_map)
    display(brand_map)
else:
    print("(nothing to show — either no OCR text, or no Overpass match within the search radii)")

## 8. Populated-places snap (in progress)

This module isn't wired into `analyze_image()` yet — shown here is just what's built so far (dataset download), as a placeholder to fill in once `load_places()` / the spatial index / snap-and-score logic are done.

In [ ]:
path = populated_places.ensure_dataset()
with open(path, encoding="utf-8") as f:
    row_count = sum(1 for _ in f)
print(f"GeoNames dataset cached at {path} ({row_count:,} rows)")
print("TODO: load_places() / PopulatedPlacesIndex / snap_and_score() not implemented yet")

## 9. Final result

Runs the real end-to-end `analyze_image()` (same function `geolocate.py`'s `main()` calls) and renders the same map `mapvis.py` would produce, inline.

In [ ]:
result = analyze_image(IMAGE_PATH, model=model, verbose=True)

points = []
for rank, cand in enumerate(result["candidates"], start=1):
    lat, lon = cand["coord"]
    points.append({
        "lat": lat, "lon": lon,
        "label": f"{IMAGE_PATH} (#{rank}, {cand['origin']}, score={cand['score']:.4f})",
        "kind": "best" if rank == 1 else "alt",
        "source": result["source"], "origin": cand["origin"],
        "spread_km": result["spread_km"], "num_clusters": result["num_clusters"],
    })
    if cand["origin"] == "ocr_brand":
        points[-1]["matched_text"] = cand["matched_text"]
        points[-1]["radius_km"] = cand["radius_km"]

final_map = mapvis.create_map(points, output_path="notebook_result_map.html", open_browser=False)
final_map